Линейные преобразования сохраняют нормальность

In [1]:
import numpy as np
from scipy.stats import normaltest

np.random.seed(1)
X = np.random.normal(size=(1000, 2))  # 2D нормальные
A = np.array([[2, 1], [-1, 3]])       # произвольная матрица
Y = X @ A.T                           # линейное преобразование

print("Проверка нормальности компонент Y:")
for i in range(2):
    stat, p = normaltest(Y[:, i])
    print(f"  Компонента {i+1}: p-value = {p:.3f}")

Проверка нормальности компонент Y:
  Компонента 1: p-value = 0.382
  Компонента 2: p-value = 0.508


Ортогональные преобразования сохраняют независимость

In [2]:
from numpy.linalg import qr

n = 5
A = np.random.normal(size=(n, n))
Q, _ = qr(A)              # получаем ортогональную матрицу
X = np.random.normal(size=n)
Y = Q @ X

print("Среднее Y:", np.mean(Y).round(3))
print("Дисперсия Y:", np.var(Y, ddof=1).round(3))
print("Проверим ортогональность Q:", np.allclose(Q @ Q.T, np.eye(n)))

Среднее Y: -0.398
Дисперсия Y: 0.746
Проверим ортогональность Q: True


Разложение на среднее и остатки

In [6]:
from numpy.linalg import qr
from scipy.stats import chi2
from scipy.stats import norm
n = 5
A = np.random.normal(size=(n, n))
Q, _ = qr(A)              # получаем ортогональную матрицу
X = np.random.normal(size=n)
Y = Q @ X
scaled = np.dot(Y, Y)
print("Среднее Y:", np.mean(Y).round(3))
print("Дисперсия Y:", np.var(Y, ddof=1).round(3))
print("Проверим ортогональность Q:", np.allclose(Q @ Q.T, np.eye(n)))
print("CDF χ²:", chi2.cdf(scaled, df=n-1))

Среднее Y: 0.488
Дисперсия Y: 1.375
Проверим ортогональность Q: True
CDF χ²: 0.8469018466417293


Проекции и матричная форма

In [7]:
n = 5
X = np.random.normal(0, 1, n)
P = np.ones((n, n)) / n
proj = P @ X
resid = X - proj

print("X:", X.round(2))
print("Проекция на среднее:", proj.round(2))
print("Остатки:", resid.round(2))
print("Проверка ортогональности:", np.isclose(np.dot(proj, resid), 0))

X: [-0.3  -0.96 -0.81 -0.57  0.58]
Проекция на среднее: [-0.41 -0.41 -0.41 -0.41 -0.41]
Остатки: [ 0.11 -0.55 -0.4  -0.16  0.99]
Проверка ортогональности: True


In [11]:
import numpy as np

# 1. Фиксируем seed
np.random.seed(12)

# 2. Генерируем выборку
n = 15
data = np.random.normal(loc=0, scale=1, size=n)

# 3. Считаем среднее и стандартное отклонение (несмещенное)
x_bar = np.mean(data)       # ~ 0.0399
s = np.std(data, ddof=1)    # ~ 1.2104

# 4. Считаем t-статистику
t_stat = (x_bar - 0) / (s / np.sqrt(n))  # ~ 0.1278

# 5. Возводим в квадрат
result = t_stat ** 2

print(f"t-статистика: {t_stat}")
print(f"t^2: {result}")
print(f"Ответ: {round(result, 2)}")

t-статистика: 0.1278183898294603
t^2: 0.016337540778595883
Ответ: 0.02


Рассмотрим на Python симуляцию покрытия t-интервала

In [12]:
import numpy as np
from scipy.stats import t, norm

def empirical_coverage_t(mu=0, sigma=1, n=10, alpha=0.05, B=5000, seed=0):
    rng = np.random.default_rng(seed)
    contains = 0
    for _ in range(B):
        x = rng.normal(loc=mu, scale=sigma, size=n)
        xbar = x.mean()
        s = x.std(ddof=1)
        t_q = t.ppf(1-alpha/2, df=n-1)
        lo = xbar - t_q * s / np.sqrt(n)
        hi = xbar + t_q * s / np.sqrt(n)
        if lo <= mu <= hi:
            contains += 1
    return contains / B

print("Empirical coverage (n=10):", empirical_coverage_t(n=10))
print("Empirical coverage (n=30):", empirical_coverage_t(n=30))

Empirical coverage (n=10): 0.9488
Empirical coverage (n=30): 0.9458


In [13]:
import numpy as np
from scipy.stats import norm, t, chi2, f

np.random.seed(42)

# Параметры
n = 20
mu_true = 2.5
sigma_true = 1.7
alpha = 0.05

# сгенерируем выборку
x = np.random.normal(loc=mu_true, scale=sigma_true, size=n)
xbar = x.mean()
s = x.std(ddof=1)

# 1) z-interval (если sigma известно)
zq = norm.ppf(1-alpha/2)
ci_z = (xbar - zq * sigma_true/np.sqrt(n), xbar + zq * sigma_true/np.sqrt(n))
print("z-CI (sigma known):", ci_z)

# 2) t-interval (sigma unknown)
tq = t.ppf(1-alpha/2, df=n-1)
ci_t = (xbar - tq * s/np.sqrt(n), xbar + tq * s/np.sqrt(n))
print("t-CI (sigma unknown):", ci_t)

# 3) CI for sigma^2 via chi2
chi_low = chi2.ppf(alpha/2, df=n-1)
chi_high = chi2.ppf(1-alpha/2, df=n-1)
ci_var = ((n-1)*s*s/chi_high, (n-1)*s*s/chi_low)
ci_sigma = (np.sqrt(ci_var[0]), np.sqrt(ci_var[1]))
print("CI for sigma^2:", ci_var)
print("CI for sigma:", ci_sigma)

# 4) Prediction interval for a single future observation
pred_halfwidth = tq * s * np.sqrt(1 + 1/n)
pred_interval = (xbar - pred_halfwidth, xbar + pred_halfwidth)
print("Prediction interval for next obs:", pred_interval)

z-CI (sigma known): (np.float64(1.4637482860587958), np.float64(2.953836605038984))
t-CI (sigma unknown): (np.float64(1.4449703214786414), np.float64(2.9726145696191386))
CI for sigma^2: (np.float64(1.5404708668001819), np.float64(5.682137628043295))
CI for sigma: (np.float64(1.2411570677396886), np.float64(2.3837234797776556))
Prediction interval for next obs: (np.float64(-1.2914802554849762), np.float64(5.709065146582756))


In [ ]:

import numpy as np
from scipy import stats

np.random.seed(42)

# 1. Генерация выборок
n1, mu1, sigma1 = 12, 2.0, 1.0
n2, mu2, sigma2 = 18, 2.5, 1.3

sample1 = np.random.normal(mu1, sigma1, n1)
sample2 = np.random.normal(mu2, sigma2, n2)

# 2. Выборочные средние и дисперсии (ddof=1)
m1, v1 = np.mean(sample1), np.var(sample1, ddof=1)
m2, v2 = np.mean(sample2), np.var(sample2, ddof=1)

# 3. Стандартная ошибка и степени свободы (Welch)
se = np.sqrt(v1/n1 + v2/n2)
df = (v1/n1 + v2/n2)**2 / ((v1/n1)**2/(n1-1) + (v2/n2)**2/(n2-1))

# 4. Доверительный интервал
alpha = 0.05
t_crit = stats.t.ppf(1 - alpha/2, df)
diff = m1 - m2

lower = diff - t_crit * se
print(f"Нижняя граница: {lower:.2f}")

Нижняя граница: 1.16


Вычисление асимптотического доверительного интервала (CI) для параметра  — для экспоненциального и пуассоновского распределений.
Симуляция покрытия интервала, чтобы проверить, насколько часто он действительно содержит истинное значение параметра.

In [17]:
import numpy as np
from scipy.stats import norm
np.random.seed(0)

z95 = norm.ppf(0.975)

def exp_lambda_ci(data, alpha=0.05):
    n = len(data)
    hat_lambda = 1.0/np.mean(data)
    se = hat_lambda/np.sqrt(n)   # plug-in SE via delta method
    lo = hat_lambda - norm.ppf(1-alpha/2)*se
    hi = hat_lambda + norm.ppf(1-alpha/2)*se
    return hat_lambda, lo, hi

def pois_lambda_ci(data, alpha=0.05):
    n = len(data)
    hat_lambda = np.mean(data)
    se = np.sqrt(hat_lambda/n)
    lo = hat_lambda - norm.ppf(1-alpha/2)*se
    hi = hat_lambda + norm.ppf(1-alpha/2)*se
    return hat_lambda, lo, hi